In [ ]:
from __future__ import annotations

import heapq
import json
import time
from itertools import combinations
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy import ndimage
from scipy.special import expit, logsumexp
from scipy.stats import beta as beta_distribution

from skimage.morphology import h_maxima
from skimage.segmentation import find_boundaries, watershed


In [ ]:
# Add project root to Python path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
SAMPLE_NAME = "44b6_0113de3b"

STAGE3_DIR = (
    PROJECT_ROOT
    / "data"
    / "sample"
    / "processed"
    / "stage3_cell_detection"
    / SAMPLE_NAME
)

binary_mask = np.load(
    STAGE3_DIR / "binary_mask.npy"
).astype(bool)

labeled = np.load(
    STAGE3_DIR / "labeled.npy"
)

objects_df = pd.read_csv(
    STAGE3_DIR / "objects.csv"
)

if binary_mask.ndim != 3:
    raise ValueError(
        f"Expected a 3-D binary mask, received shape {binary_mask.shape}."
    )

print("Binary mask shape:", binary_mask.shape)
print("Labeled shape:", labeled.shape)
print("Stage 3 objects:", len(objects_df))


## Distance Transform

In [ ]:
VOXEL_SIZE = np.asarray(
    (
        1.625,    # z (µm)
        0.40625,  # y (µm)
        0.40625,  # x (µm)
    ),
    dtype=float,
)

distance = ndimage.distance_transform_edt(
    binary_mask,
    sampling=VOXEL_SIZE,
).astype(np.float32)

print("Distance transform")
print("Shape:", distance.shape)
print("Max distance:", float(distance.max()))


In [ ]:
z = distance.shape[0] // 2

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(binary_mask[z], cmap="gray")
plt.title(f"Binary Mask (z={z})")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(distance[z], cmap="viridis")
plt.title(f"Distance Transform (z={z})")
plt.axis("off")

plt.tight_layout()
plt.show()


### 3-D to 2-D maximum projection

In [ ]:
distance_mip = distance.max(axis=0)

plt.figure(figsize=(6, 6))
plt.imshow(distance_mip, cmap="viridis")
plt.title("Distance Transform (Maximum Projection)")
plt.axis("off")
plt.colorbar()
plt.show()


## Label connected components

In [ ]:
component_labels, num_components = ndimage.label(
    binary_mask,
    structure=ndimage.generate_binary_structure(3, 1),
)

component_slices = ndimage.find_objects(component_labels)

print("Connected components:", num_components)
print("Background label:", int(component_labels.min()))
print("Largest label:", int(component_labels.max()))


The global smoothed EDT is retained for visualization and safe fallback markers. Final instance decisions are made independently inside each connected component using the probabilistic model below.

In [ ]:
# Retained for the original notebook visualizations and as a robust fallback
# marker field. Probabilistic splitting computes component-local EDT fields.
sigma_physical = 0.8

sigma_zyx = sigma_physical / VOXEL_SIZE

distance_smooth = ndimage.gaussian_filter(
    distance,
    sigma=sigma_zyx,
    mode="nearest",
).astype(np.float32)

print("Smoothed EDT maximum:", float(distance_smooth.max()))


## Probabilistic component-level instance segmentation

The original notebook used one global Gaussian-smoothed EDT, one fixed
`h_maxima` threshold, and one watershed pass. The integrated pipeline keeps
the existing data loading, EDT visualizations, output arrays, boundary
inspection, size statistics, and save locations, but replaces the fixed marker
decision with the validated spatial probabilistic splitter.

### Implementation plan

1. **Process each connected component independently.**  
   A padded local crop keeps the merge-tree calculations efficient and prevents
   unrelated cells from influencing one another.

2. **Generate liberal marker candidates.**  
   EDT maxima are collected across several physical smoothing scales and
   prominence levels, then clustered in physical coordinates.

3. **Distinguish maxima from biological lobes.**  
   For every candidate pair, the algorithm measures the widest-path EDT saddle,
   branch persistence, branch-owned volume, physical separation, and
   multi-scale peak support. Beta likelihood models estimate
   `P(distinct lobes)` versus `P(same lobe)`.

4. **Collapse duplicate centers inside one cell.**  
   Complete-link grouping merges maxima that are probabilistically likely to
   belong to the same EDT lobe. This directly prevents two detected centers
   inside one individual cell from automatically producing two instances.

5. **Evaluate hierarchical hypotheses.**  
   `H1` retains one cell. `H2` is the primary split model. `H3` is generated
   only when three independently supported lobes survive and must beat `H2`
   by stronger posterior odds.

6. **Use broad hard gates only for safety.**  
   Hard gates reject incomplete partitions, disconnected children,
   sub-resolution marker spacing, and numerical fragments. Shape, neck,
   balance, and prominence remain smooth probabilistic evidence.

7. **Keep uncertain components unsplit.**  
   When a split does not clear the posterior acceptance rule, the original
   component is preserved and recorded as `uncertain_no_split`.

8. **Preserve pipeline outputs and add diagnostics.**  
   Final marker IDs are aligned with final instance IDs. The notebook also
   records per-component posterior decisions, hypothesis evidence, and the
   mapping from Stage 3 components to Stage 4 instances.


In [ ]:
# ============================================================
# Spatial-only probabilistic split configuration
# ============================================================

SPLIT_CONFIG = {
    # Liberal multi-scale peak generation.
    "sigma_levels_um": (0.25, 0.40, 0.60, 0.85, 1.10),
    "h_levels_um": (0.10, 0.18, 0.28, 0.42, 0.60),
    "peak_cluster_radius_um": 1.10,
    "max_candidate_peaks": 10,
    "max_combinations_per_k": 60,

    # Light smoothing for merge-tree topology and stable watershed boundaries.
    "merge_tree_sigma_um": 0.20,
    "watershed_sigma_um": 0.50,

    # Optimize for the common two-cell merge; retain rare three-cell support.
    "max_cells": 3,

    # Broad safety hard gates only.
    "hard_min_marker_separation_um": 0.85,
    "hard_min_child_voxels": 24,
    "hard_min_child_fraction": {2: 0.025, 3: 0.020},
    "hard_min_equivalent_radius_um": 0.55,

    # Collapse maxima likely to be multiple centers inside the same EDT lobe.
    "same_lobe_collapse_probability": 0.76,

    # Pairwise distinct-lobe versus same-lobe probability model.
    "pair_prior_distinct": 0.35,
    "pair_likelihood_temperature": 1.60,
    "pair_feature_models": {
        "branch_persistence": {
            "distinct": (4.0, 2.2),
            "same": (1.5, 5.0),
            "weight": 1.35,
        },
        "branch_balance": {
            "distinct": (3.0, 2.7),
            "same": (1.2, 6.0),
            "weight": 1.15,
        },
        "separation_support": {
            "distinct": (4.0, 2.2),
            "same": (2.0, 4.2),
            "weight": 0.75,
        },
        "peak_support": {
            "distinct": (3.5, 1.9),
            "same": (2.0, 2.8),
            "weight": 0.45,
        },
    },

    # These are the curated suspicious-component priors. They are used only
    # after lobe collapsing leaves competing spatial hypotheses. A component
    # with one effective lobe can generate H1 only.
    "hypothesis_priors": {1: 0.24, 2: 0.66, 3: 0.10},

    "hypothesis_evidence_weights": {
        "lobe_support": 1.35,
        "coverage_support": 2.50,
        "marker_quality": 0.35,
        "neck_support": 1.05,
        "child_shape": 0.85,
        "shape_improvement": 0.75,
        "child_volume": 0.55,
        "fragment_safety": 1.10,
    },

    # H3 is generated only with three independently supported lobes.
    "k3_generation_min_pair_probability": 0.22,
    "k3_generation_min_geometric_probability": 0.40,

    # Hierarchical posterior acceptance.
    "h2_min_conditional_probability": 0.62,
    "h2_min_odds_vs_h1": 1.65,
    "h3_min_conditional_probability": 0.78,
    "h3_min_odds_vs_h2": 3.50,
    "h1_confident_conditional_probability": 0.38,

    "probability_epsilon": 1e-6,

    # Whole-volume integration settings.
    "component_padding_voxels": 2,
    "progress_every_components": 25,
}

SPLIT_CONFIG


In [ ]:
# ============================================================
# Translation-independent 3-D cell-shape properties
# ============================================================

def describe_cell_mask(
    mask: np.ndarray,
    voxel_size_zyx: np.ndarray,
) -> dict:
    mask = np.asarray(mask, dtype=bool)
    voxel_size = np.asarray(voxel_size_zyx, dtype=float)
    coordinates_zyx = np.argwhere(mask)

    if len(coordinates_zyx) == 0:
        raise ValueError("The cell mask is empty.")

    physical_coordinates = (
        coordinates_zyx.astype(float)
        * voxel_size[None, :]
    )
    centroid_physical = physical_coordinates.mean(axis=0)
    centered_coordinates = (
        physical_coordinates
        - centroid_physical[None, :]
    )

    if len(coordinates_zyx) >= 4:
        covariance = np.cov(
            centered_coordinates,
            rowvar=False,
            bias=True,
        )
        eigenvalues, eigenvectors = np.linalg.eigh(covariance)
        descending_order = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[descending_order]
        eigenvectors = eigenvectors[:, descending_order]

        projected_coordinates = centered_coordinates @ eigenvectors
        lower_percentile = np.percentile(
            projected_coordinates,
            2.5,
            axis=0,
        )
        upper_percentile = np.percentile(
            projected_coordinates,
            97.5,
            axis=0,
        )
        principal_extents_um = upper_percentile - lower_percentile
    else:
        # Tiny Stage 3 components are preserved as one instance. Their bbox
        # dimensions provide a stable fallback descriptor for diagnostics.
        minimum_coordinates = coordinates_zyx.min(axis=0)
        maximum_coordinates = coordinates_zyx.max(axis=0)
        bbox_dimensions_um = (
            maximum_coordinates - minimum_coordinates + 1
        ) * voxel_size
        descending_order = np.argsort(bbox_dimensions_um)[::-1]
        principal_extents_um = bbox_dimensions_um[descending_order]
        eigenvectors = np.eye(3)[:, descending_order]
        eigenvalues = np.zeros(3, dtype=float)

    major_axis_um = float(principal_extents_um[0])
    middle_axis_um = float(principal_extents_um[1])
    minor_axis_um = float(principal_extents_um[2])

    voxel_count = int(mask.sum())
    volume_um3 = float(voxel_count * np.prod(voxel_size))
    equivalent_radius_um = float(
        (3.0 * volume_um3 / (4.0 * np.pi)) ** (1.0 / 3.0)
    )

    minimum_coordinates = coordinates_zyx.min(axis=0)
    maximum_coordinates = coordinates_zyx.max(axis=0)
    bbox_shape_voxels = (
        maximum_coordinates - minimum_coordinates + 1
    )
    bbox_dimensions_um = bbox_shape_voxels * voxel_size
    bbox_voxel_count = int(np.prod(bbox_shape_voxels))
    extent = (
        voxel_count / bbox_voxel_count
        if bbox_voxel_count > 0
        else np.nan
    )

    denominator_floor = 0.5 * float(voxel_size.min())
    safe_minor_axis_um = max(minor_axis_um, denominator_floor)
    elongation = major_axis_um / safe_minor_axis_um
    flatness = middle_axis_um / safe_minor_axis_um

    return {
        "voxel_count": voxel_count,
        "volume_um3": volume_um3,
        "equivalent_radius_um": equivalent_radius_um,
        "major_axis_um": major_axis_um,
        "middle_axis_um": middle_axis_um,
        "minor_axis_um": minor_axis_um,
        "elongation": float(elongation),
        "flatness": float(flatness),
        "extent": float(extent),
        "bbox_depth_um": float(bbox_dimensions_um[0]),
        "bbox_height_um": float(bbox_dimensions_um[1]),
        "bbox_width_um": float(bbox_dimensions_um[2]),
        "centroid_z_um": float(centroid_physical[0]),
        "centroid_y_um": float(centroid_physical[1]),
        "centroid_x_um": float(centroid_physical[2]),
        "principal_vectors_zyx": eigenvectors,
        "principal_variances": eigenvalues,
    }

# ============================================================
# Multi-scale persistent EDT peak generation
# ============================================================

def physical_sigma_voxels(
    sigma_um: float,
    voxel_size_zyx: np.ndarray,
) -> np.ndarray:
    """Convert an isotropic physical Gaussian width into voxel units."""

    voxel_size = np.asarray(voxel_size_zyx, dtype=float)

    if voxel_size.shape != (3,) or np.any(voxel_size <= 0):
        raise ValueError(
            "voxel_size_zyx must contain three positive values."
        )

    return float(sigma_um) / voxel_size


def physical_distance_between_points(
    point_a_zyx: np.ndarray,
    point_b_zyx: np.ndarray,
    voxel_size_zyx: np.ndarray,
) -> float:
    delta = (
        np.asarray(point_a_zyx, dtype=float)
        - np.asarray(point_b_zyx, dtype=float)
    )

    return float(
        np.linalg.norm(
            delta * np.asarray(voxel_size_zyx, dtype=float)
        )
    )


def _peak_position_from_record(
    peak_record: dict,
) -> tuple[int, int, int]:
    return (
        int(peak_record["z"]),
        int(peak_record["y"]),
        int(peak_record["x"]),
    )


def _representative_peak_position(
    peak_component: np.ndarray,
    smoothed_distance: np.ndarray,
) -> tuple[int, int, int]:
    coordinates = np.argwhere(peak_component)

    if len(coordinates) == 0:
        raise ValueError("Peak component is empty.")

    values = smoothed_distance[tuple(coordinates.T)]

    return tuple(
        int(value)
        for value in coordinates[int(np.argmax(values))]
    )


def _cluster_peak_detections(
    detections: list[dict],
    voxel_size_zyx: np.ndarray,
    cluster_radius_um: float,
) -> list[list[dict]]:
    """Greedily merge repeated detections of the same physical maximum."""

    ordered = sorted(
        detections,
        key=lambda record: (
            record["raw_depth_um"],
            record["smoothed_depth_um"],
        ),
        reverse=True,
    )

    clusters: list[list[dict]] = []

    for detection in ordered:
        point = np.asarray(detection["position_zyx"], dtype=float)

        best_index = None
        best_distance = np.inf

        for index, cluster in enumerate(clusters):
            representative = np.asarray(
                max(
                    cluster,
                    key=lambda record: (
                        record["raw_depth_um"],
                        record["smoothed_depth_um"],
                    ),
                )["position_zyx"],
                dtype=float,
            )

            distance = physical_distance_between_points(
                point,
                representative,
                voxel_size_zyx,
            )

            if distance < best_distance:
                best_index = index
                best_distance = distance

        if (
            best_index is not None
            and best_distance <= float(cluster_radius_um)
        ):
            clusters[best_index].append(detection)
        else:
            clusters.append([detection])

    return clusters


def detect_persistent_distance_peaks(
    mask: np.ndarray,
    voxel_size_zyx: np.ndarray,
    config: dict,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, pd.DataFrame]:
    """Return raw, merge-tree and watershed EDTs plus peak candidates."""

    mask = np.asarray(mask, dtype=bool)

    if mask.ndim != 3 or not mask.any():
        raise ValueError("mask must be a non-empty 3-D binary array.")

    voxel_size = np.asarray(voxel_size_zyx, dtype=float)

    raw_distance = ndimage.distance_transform_edt(
        mask,
        sampling=voxel_size,
    ).astype(np.float32)

    detections: list[dict] = []

    sigma_levels = tuple(
        float(value)
        for value in config["sigma_levels_um"]
    )
    h_levels = tuple(
        float(value)
        for value in config["h_levels_um"]
    )

    for sigma_um in sigma_levels:
        smoothed_distance = ndimage.gaussian_filter(
            raw_distance,
            sigma=physical_sigma_voxels(
                sigma_um,
                voxel_size,
            ),
            mode="nearest",
        )

        for h_um in h_levels:
            maxima_mask = h_maxima(
                smoothed_distance,
                h=h_um,
            ) & mask

            peak_components, component_count = ndimage.label(
                maxima_mask,
                structure=ndimage.generate_binary_structure(3, 1),
            )

            for component_id in range(1, component_count + 1):
                component = peak_components == component_id

                position = _representative_peak_position(
                    component,
                    smoothed_distance,
                )

                detections.append(
                    {
                        "position_zyx": position,
                        "sigma_um": sigma_um,
                        "h_um": h_um,
                        "raw_depth_um": float(
                            raw_distance[position]
                        ),
                        "smoothed_depth_um": float(
                            smoothed_distance[position]
                        ),
                    }
                )

    if not detections:
        global_peak = tuple(
            int(value)
            for value in np.unravel_index(
                int(np.argmax(raw_distance)),
                raw_distance.shape,
            )
        )

        detections.append(
            {
                "position_zyx": global_peak,
                "sigma_um": 0.0,
                "h_um": 0.0,
                "raw_depth_um": float(raw_distance[global_peak]),
                "smoothed_depth_um": float(raw_distance[global_peak]),
            }
        )

    clusters = _cluster_peak_detections(
        detections=detections,
        voxel_size_zyx=voxel_size,
        cluster_radius_um=float(
            config["peak_cluster_radius_um"]
        ),
    )

    maximum_distance = max(float(raw_distance.max()), 1e-6)
    peak_records: list[dict] = []

    for peak_id, cluster in enumerate(clusters, start=1):
        representative = max(
            cluster,
            key=lambda record: (
                record["raw_depth_um"],
                record["smoothed_depth_um"],
            ),
        )

        unique_sigmas = {
            float(record["sigma_um"])
            for record in cluster
        }
        unique_h = {
            float(record["h_um"])
            for record in cluster
        }
        unique_settings = {
            (
                float(record["sigma_um"]),
                float(record["h_um"]),
            )
            for record in cluster
        }

        scale_support = len(unique_sigmas) / max(len(sigma_levels), 1)
        h_support = len(unique_h) / max(len(h_levels), 1)
        setting_support = (
            len(unique_settings)
            / max(len(sigma_levels) * len(h_levels), 1)
        )

        depth_score = (
            float(representative["raw_depth_um"])
            / maximum_distance
        )

        persistence_score = float(
            0.35 * scale_support
            + 0.25 * h_support
            + 0.25 * depth_score
            + 0.15 * np.sqrt(setting_support)
        )

        position = tuple(
            int(value)
            for value in representative["position_zyx"]
        )

        peak_records.append(
            {
                "peak_id": int(peak_id),
                "z": position[0],
                "y": position[1],
                "x": position[2],
                "raw_depth_um": float(
                    representative["raw_depth_um"]
                ),
                "smoothed_depth_um": float(
                    representative["smoothed_depth_um"]
                ),
                "scale_support": float(scale_support),
                "h_support": float(h_support),
                "setting_support": float(setting_support),
                "detection_count": int(len(cluster)),
                "persistence_score": persistence_score,
            }
        )

    peak_table = (
        pd.DataFrame(peak_records)
        .sort_values(
            ["persistence_score", "raw_depth_um"],
            ascending=False,
        )
        .reset_index(drop=True)
    )

    peak_table["rank"] = np.arange(
        len(peak_table),
        dtype=int,
    ) + 1

    merge_tree_distance = ndimage.gaussian_filter(
        raw_distance,
        sigma=physical_sigma_voxels(
            float(config["merge_tree_sigma_um"]),
            voxel_size,
        ),
        mode="nearest",
    ).astype(np.float32)

    watershed_distance = ndimage.gaussian_filter(
        raw_distance,
        sigma=physical_sigma_voxels(
            float(config["watershed_sigma_um"]),
            voxel_size,
        ),
        mode="nearest",
    ).astype(np.float32)

    return (
        raw_distance,
        merge_tree_distance,
        watershed_distance,
        peak_table,
    )

In [ ]:
# ============================================================
# EDT merge-tree pair evidence and same-lobe collapsing
# ============================================================

_NEIGHBOR_OFFSETS_6 = (
    (-1, 0, 0),
    (1, 0, 0),
    (0, -1, 0),
    (0, 1, 0),
    (0, 0, -1),
    (0, 0, 1),
)


def widest_path_saddle_level(
    distance_um: np.ndarray,
    mask: np.ndarray,
    start_zyx: tuple[int, int, int],
    end_zyx: tuple[int, int, int],
) -> float:
    """Maximum possible minimum EDT value along a 6-connected path."""

    distance = np.asarray(distance_um, dtype=float)
    mask = np.asarray(mask, dtype=bool)

    if not mask[start_zyx] or not mask[end_zyx]:
        raise ValueError("Both peak positions must lie inside the mask.")

    best = np.full(distance.shape, -np.inf, dtype=np.float32)
    start_value = float(distance[start_zyx])
    best[start_zyx] = start_value

    queue: list[tuple[float, tuple[int, int, int]]] = [
        (-start_value, start_zyx)
    ]

    shape = distance.shape

    while queue:
        negative_score, current = heapq.heappop(queue)
        current_score = -float(negative_score)

        if current == end_zyx:
            return current_score

        if current_score < float(best[current]) - 1e-8:
            continue

        z, y, x = current

        for dz, dy, dx in _NEIGHBOR_OFFSETS_6:
            neighbor = (z + dz, y + dy, x + dx)

            if not (
                0 <= neighbor[0] < shape[0]
                and 0 <= neighbor[1] < shape[1]
                and 0 <= neighbor[2] < shape[2]
            ):
                continue

            if not mask[neighbor]:
                continue

            candidate = min(
                current_score,
                float(distance[neighbor]),
            )

            if candidate > float(best[neighbor]) + 1e-8:
                best[neighbor] = candidate
                heapq.heappush(
                    queue,
                    (-candidate, neighbor),
                )

    raise RuntimeError(
        "Peak candidates are not connected inside the selected component."
    )


def branch_size_above_saddle(
    distance_um: np.ndarray,
    mask: np.ndarray,
    peak_zyx: tuple[int, int, int],
    saddle_um: float,
) -> int:
    """Size of the peak-owned high-EDT branch immediately above a saddle."""

    epsilon = max(
        np.finfo(np.float32).eps,
        1e-5 * max(float(distance_um[peak_zyx]), 1.0),
    )

    branch_mask = (
        np.asarray(mask, dtype=bool)
        & (np.asarray(distance_um, dtype=float) > float(saddle_um) + epsilon)
    )

    if not branch_mask[peak_zyx]:
        return 1

    labels, _ = ndimage.label(
        branch_mask,
        structure=ndimage.generate_binary_structure(3, 1),
    )

    label = int(labels[peak_zyx])

    if label <= 0:
        return 1

    return int(np.count_nonzero(labels == label))


def _safe_beta_logpdf(
    value: float,
    parameters: tuple[float, float],
    epsilon: float,
) -> float:
    clipped = float(np.clip(value, epsilon, 1.0 - epsilon))
    a, b = (float(parameters[0]), float(parameters[1]))

    return float(beta_distribution.logpdf(clipped, a, b))


def pair_distinct_lobe_probability(
    transformed_features: dict[str, float],
    config: dict,
) -> tuple[float, float]:
    """Posterior P(distinct lobes | pairwise EDT branch features)."""

    epsilon = float(config["probability_epsilon"])
    prior = float(config["pair_prior_distinct"])

    log_odds = float(
        np.log(prior + epsilon)
        - np.log(1.0 - prior + epsilon)
    )

    for feature_name, model in config[
        "pair_feature_models"
    ].items():
        value = float(transformed_features[feature_name])
        weight = float(model["weight"])

        distinct_logpdf = _safe_beta_logpdf(
            value,
            tuple(model["distinct"]),
            epsilon,
        )
        same_logpdf = _safe_beta_logpdf(
            value,
            tuple(model["same"]),
            epsilon,
        )

        log_odds += weight * (
            distinct_logpdf - same_logpdf
        )

    log_odds /= max(
        float(config["pair_likelihood_temperature"]),
        epsilon,
    )

    posterior = float(expit(log_odds))

    return posterior, float(log_odds)


def build_peak_pair_table(
    peak_table: pd.DataFrame,
    mask: np.ndarray,
    merge_tree_distance: np.ndarray,
    voxel_size_zyx: np.ndarray,
    config: dict,
) -> pd.DataFrame:
    records = peak_table.head(
        int(config["max_candidate_peaks"])
    ).to_dict("records")

    pair_records: list[dict] = []
    total_voxels = max(int(np.count_nonzero(mask)), 1)

    for first, second in combinations(records, 2):
        first_position = _peak_position_from_record(first)
        second_position = _peak_position_from_record(second)

        first_depth = float(
            merge_tree_distance[first_position]
        )
        second_depth = float(
            merge_tree_distance[second_position]
        )

        smaller_depth = max(
            min(first_depth, second_depth),
            1e-6,
        )

        saddle_um = widest_path_saddle_level(
            distance_um=merge_tree_distance,
            mask=mask,
            start_zyx=first_position,
            end_zyx=second_position,
        )

        branch_persistence = float(
            np.clip(
                1.0 - saddle_um / smaller_depth,
                0.0,
                1.0,
            )
        )

        first_branch_voxels = branch_size_above_saddle(
            merge_tree_distance,
            mask,
            first_position,
            saddle_um,
        )
        second_branch_voxels = branch_size_above_saddle(
            merge_tree_distance,
            mask,
            second_position,
            saddle_um,
        )

        smaller_branch_fraction = float(
            min(first_branch_voxels, second_branch_voxels)
            / total_voxels
        )

        # Branch cores immediately above a saddle are naturally much smaller
        # than the full component. A fraction around 8% already represents a
        # substantial independent high-EDT branch, so it maps near one.
        branch_balance = float(
            np.clip(
                smaller_branch_fraction / 0.08,
                0.0,
                1.0,
            )
        )

        separation_um = physical_distance_between_points(
            first_position,
            second_position,
            voxel_size_zyx,
        )

        separation_ratio = float(
            separation_um
            / max(first_depth + second_depth, 1e-6)
        )

        separation_support = float(
            separation_ratio / (separation_ratio + 0.65)
        )

        peak_support = float(
            np.sqrt(
                max(float(first["persistence_score"]), 0.0)
                * max(float(second["persistence_score"]), 0.0)
            )
        )

        transformed = {
            "branch_persistence": branch_persistence,
            "branch_balance": branch_balance,
            "separation_support": separation_support,
            "peak_support": peak_support,
        }

        distinct_probability, distinct_log_odds = (
            pair_distinct_lobe_probability(
                transformed,
                config,
            )
        )

        pair_records.append(
            {
                "peak_id_a": int(first["peak_id"]),
                "peak_id_b": int(second["peak_id"]),
                "separation_um": separation_um,
                "separation_ratio": separation_ratio,
                "peak_depth_a_um": first_depth,
                "peak_depth_b_um": second_depth,
                "saddle_um": float(saddle_um),
                "saddle_ratio": float(
                    saddle_um / smaller_depth
                ),
                "branch_persistence": branch_persistence,
                "branch_voxels_a": first_branch_voxels,
                "branch_voxels_b": second_branch_voxels,
                "smaller_branch_fraction": smaller_branch_fraction,
                "branch_balance": branch_balance,
                "separation_support": separation_support,
                "peak_support": peak_support,
                "distinct_lobe_log_odds": distinct_log_odds,
                "distinct_lobe_probability": distinct_probability,
                "same_lobe_probability": 1.0 - distinct_probability,
            }
        )

    if not pair_records:
        return pd.DataFrame(
            columns=[
                "peak_id_a",
                "peak_id_b",
                "distinct_lobe_probability",
                "same_lobe_probability",
            ]
        )

    return (
        pd.DataFrame(pair_records)
        .sort_values(
            "distinct_lobe_probability",
            ascending=False,
        )
        .reset_index(drop=True)
    )


def collapse_same_lobe_peaks(
    peak_table: pd.DataFrame,
    pair_table: pd.DataFrame,
    config: dict,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Complete-link grouping avoids transitive same-lobe chain collapse."""

    candidate_table = (
        peak_table
        .head(int(config["max_candidate_peaks"]))
        .copy()
        .reset_index(drop=True)
    )

    same_lookup = {
        frozenset(
            (
                int(record["peak_id_a"]),
                int(record["peak_id_b"]),
            )
        ): float(record["same_lobe_probability"])
        for record in pair_table.to_dict("records")
    }

    ordered_records = sorted(
        candidate_table.to_dict("records"),
        key=lambda record: (
            float(record["persistence_score"])
            * float(record["raw_depth_um"]),
            float(record["raw_depth_um"]),
        ),
        reverse=True,
    )

    threshold = float(
        config["same_lobe_collapse_probability"]
    )

    groups: list[list[dict]] = []

    for record in ordered_records:
        peak_id = int(record["peak_id"])
        compatible_group = None
        best_minimum_probability = -np.inf

        for group_index, group in enumerate(groups):
            probabilities = [
                same_lookup.get(
                    frozenset(
                        (
                            peak_id,
                            int(member["peak_id"]),
                        )
                    ),
                    0.0,
                )
                for member in group
            ]

            minimum_probability = (
                min(probabilities)
                if probabilities
                else 1.0
            )

            if (
                minimum_probability >= threshold
                and minimum_probability
                > best_minimum_probability
            ):
                compatible_group = group_index
                best_minimum_probability = minimum_probability

        if compatible_group is None:
            groups.append([record])
        else:
            groups[compatible_group].append(record)

    lobe_id_by_peak: dict[int, int] = {}
    representative_ids: set[int] = set()

    for lobe_id, group in enumerate(groups, start=1):
        representative = max(
            group,
            key=lambda record: (
                float(record["persistence_score"])
                * float(record["raw_depth_um"]),
                float(record["raw_depth_um"]),
            ),
        )

        representative_ids.add(
            int(representative["peak_id"])
        )

        for record in group:
            lobe_id_by_peak[
                int(record["peak_id"])
            ] = int(lobe_id)

    candidate_table["effective_lobe_id"] = (
        candidate_table["peak_id"]
        .map(lobe_id_by_peak)
        .astype(int)
    )
    candidate_table["is_effective_representative"] = (
        candidate_table["peak_id"]
        .astype(int)
        .isin(representative_ids)
    )
    candidate_table["same_lobe_cluster_size"] = (
        candidate_table["effective_lobe_id"]
        .map(
            candidate_table[
                "effective_lobe_id"
            ].value_counts()
        )
        .astype(int)
    )

    effective_table = (
        candidate_table[
            candidate_table["is_effective_representative"]
        ]
        .sort_values(
            ["persistence_score", "raw_depth_um"],
            ascending=False,
        )
        .reset_index(drop=True)
    )

    effective_table["effective_rank"] = (
        np.arange(len(effective_table), dtype=int) + 1
    )

    return candidate_table, effective_table

In [ ]:
# ============================================================
# Region, interface and broad safety measurements
# ============================================================

def region_surface_area_um2(
    region_mask: np.ndarray,
    voxel_size_zyx: np.ndarray,
) -> float:
    region = np.asarray(region_mask, dtype=bool)
    voxel_size = np.asarray(voxel_size_zyx, dtype=float)

    total_area = 0.0

    for axis in range(3):
        padded = np.pad(
            region.astype(np.int8),
            [
                (1, 1) if index == axis else (0, 0)
                for index in range(3)
            ],
            mode="constant",
        )

        transitions = np.diff(
            padded,
            axis=axis,
        ) != 0

        face_area = float(
            np.prod(np.delete(voxel_size, axis))
        )

        total_area += (
            int(np.count_nonzero(transitions))
            * face_area
        )

    return float(total_area)


def pair_interface_statistics(
    labels: np.ndarray,
    label_a: int,
    label_b: int,
    distance_um: np.ndarray,
    voxel_size_zyx: np.ndarray,
) -> dict:
    labels = np.asarray(labels)
    distance_um = np.asarray(distance_um, dtype=float)
    voxel_size = np.asarray(voxel_size_zyx, dtype=float)

    interface_values: list[np.ndarray] = []
    interface_points: list[np.ndarray] = []
    interface_area_um2 = 0.0

    for axis in range(3):
        first_slice = [slice(None)] * 3
        second_slice = [slice(None)] * 3

        first_slice[axis] = slice(0, -1)
        second_slice[axis] = slice(1, None)

        first_values = labels[tuple(first_slice)]
        second_values = labels[tuple(second_slice)]

        touching = (
            (
                (first_values == label_a)
                & (second_values == label_b)
            )
            | (
                (first_values == label_b)
                & (second_values == label_a)
            )
        )

        if not np.any(touching):
            continue

        first_coordinates = np.argwhere(touching)
        second_coordinates = first_coordinates.copy()
        second_coordinates[:, axis] += 1

        first_distance = distance_um[
            tuple(first_coordinates.T)
        ]
        second_distance = distance_um[
            tuple(second_coordinates.T)
        ]

        interface_values.append(
            np.minimum(first_distance, second_distance)
        )

        interface_points.append(
            0.5 * (
                first_coordinates.astype(float)
                + second_coordinates.astype(float)
            )
        )

        face_area = float(
            np.prod(np.delete(voxel_size, axis))
        )

        interface_area_um2 += (
            int(np.count_nonzero(touching))
            * face_area
        )

    if not interface_values:
        return {
            "contact": False,
            "interface_area_um2": 0.0,
            "distance_median_um": 0.0,
            "distance_mean_um": 0.0,
            "interface_points_zyx": np.empty((0, 3), dtype=float),
        }

    values = np.concatenate(interface_values)
    points = np.concatenate(interface_points, axis=0)

    return {
        "contact": True,
        "interface_area_um2": float(interface_area_um2),
        "distance_median_um": float(np.median(values)),
        "distance_mean_um": float(np.mean(values)),
        "interface_points_zyx": points,
    }


def describe_hypothesis_regions(
    labels: np.ndarray,
    voxel_size_zyx: np.ndarray,
) -> dict[int, dict]:
    descriptions: dict[int, dict] = {}

    for label in sorted(
        int(value)
        for value in np.unique(labels)
        if int(value) > 0
    ):
        region_mask = labels == label
        description = describe_cell_mask(
            region_mask,
            voxel_size_zyx,
        )
        description["surface_area_um2"] = (
            region_surface_area_um2(
                region_mask,
                voxel_size_zyx,
            )
        )
        descriptions[label] = description

    return descriptions


def build_watershed_hypothesis(
    mask: np.ndarray,
    watershed_distance: np.ndarray,
    selected_peaks: list[dict],
) -> np.ndarray:
    marker_image = np.zeros(
        mask.shape,
        dtype=np.int32,
    )

    for marker_label, peak_record in enumerate(
        selected_peaks,
        start=1,
    ):
        position = _peak_position_from_record(peak_record)

        if not mask[position]:
            raise ValueError(
                f"Marker {position} is outside the merged mask."
            )

        marker_image[position] = marker_label

    return watershed(
        -watershed_distance,
        markers=marker_image,
        mask=mask,
        watershed_line=False,
    ).astype(np.int32)


def hard_gate_hypothesis(
    labels: np.ndarray,
    mask: np.ndarray,
    selected_peaks: list[dict],
    voxel_size_zyx: np.ndarray,
    config: dict,
) -> tuple[bool, list[str]]:
    """Reject only invalid partitions or unsafe split children.

    H1 is always allowed to preserve a non-empty Stage 3 component, including
    tiny components. Minimum child-size and resolution gates apply only to
    actual split hypotheses.
    """

    labels = np.asarray(labels)
    mask = np.asarray(mask, dtype=bool)
    reasons: list[str] = []

    if labels.shape != mask.shape:
        return False, ["shape_mismatch"]

    if np.any(labels[~mask] != 0):
        reasons.append("labels_outside_mask")

    if np.any(labels[mask] <= 0):
        reasons.append("mask_not_fully_partitioned")

    positive_labels = [
        int(value)
        for value in np.unique(labels)
        if int(value) > 0
    ]
    expected_k = len(selected_peaks)

    if len(positive_labels) != expected_k:
        reasons.append("wrong_child_count")

    if expected_k > 1:
        minimum_separation = min(
            physical_distance_between_points(
                _peak_position_from_record(first),
                _peak_position_from_record(second),
                voxel_size_zyx,
            )
            for first, second in combinations(selected_peaks, 2)
        )

        if minimum_separation < float(
            config["hard_min_marker_separation_um"]
        ):
            reasons.append("sub_resolution_marker_spacing")

    total_voxels = max(int(np.count_nonzero(mask)), 1)
    minimum_fraction = float(
        config["hard_min_child_fraction"].get(expected_k, 0.015)
    )
    connectivity = ndimage.generate_binary_structure(3, 1)

    for label_value in positive_labels:
        region = labels == label_value
        voxel_count = int(np.count_nonzero(region))

        _, component_count = ndimage.label(
            region,
            structure=connectivity,
        )
        if int(component_count) != 1:
            reasons.append(f"child_{label_value}_disconnected")

        if expected_k <= 1:
            continue

        if voxel_count < int(config["hard_min_child_voxels"]):
            reasons.append(f"child_{label_value}_too_small")
            continue

        if voxel_count / total_voxels < minimum_fraction:
            reasons.append(f"child_{label_value}_fraction_too_small")

        try:
            radius_um = float(
                describe_cell_mask(
                    region,
                    voxel_size_zyx,
                )["equivalent_radius_um"]
            )
        except ValueError:
            reasons.append(f"child_{label_value}_not_measurable")
            continue

        if radius_um < float(
            config["hard_min_equivalent_radius_um"]
        ):
            reasons.append(f"child_{label_value}_sub_resolution")

    for marker_label, peak_record in enumerate(
        selected_peaks,
        start=1,
    ):
        position = _peak_position_from_record(peak_record)

        if not mask[position]:
            reasons.append(f"marker_{marker_label}_outside_mask")
            continue

        if int(labels[position]) != marker_label:
            reasons.append(f"marker_{marker_label}_label_mismatch")

    return len(reasons) == 0, reasons


In [ ]:
# ============================================================
# Prior-informed hypothesis probabilities
# ============================================================

def _geometric_mean_probability(
    values: list[float] | np.ndarray,
    epsilon: float,
) -> float:
    values = np.asarray(values, dtype=float)

    if values.size == 0:
        return 0.5

    values = np.clip(values, epsilon, 1.0)

    return float(np.exp(np.mean(np.log(values))))


def _pair_probability_lookup(
    pair_table: pd.DataFrame,
) -> dict[frozenset[int], float]:
    return {
        frozenset(
            (
                int(record["peak_id_a"]),
                int(record["peak_id_b"]),
            )
        ): float(record["distinct_lobe_probability"])
        for record in pair_table.to_dict("records")
    }


def selected_pair_probabilities(
    selected_peaks: list[dict],
    pair_lookup: dict[frozenset[int], float],
) -> list[float]:
    probabilities: list[float] = []

    for first, second in combinations(selected_peaks, 2):
        key = frozenset(
            (
                int(first["peak_id"]),
                int(second["peak_id"]),
            )
        )
        probabilities.append(
            float(pair_lookup.get(key, 0.0))
        )

    return probabilities


def single_cell_shape_probability(
    description: dict,
) -> float:
    """Broad probability that one region remains a plausible single cell."""

    elongation_probability = float(
        expit(
            (
                4.10 - float(description["elongation"])
            )
            / 0.80
        )
    )
    flatness_probability = float(
        expit(
            (
                3.10 - float(description["flatness"])
            )
            / 0.65
        )
    )
    extent_probability = float(
        expit(
            (
                float(description["extent"]) - 0.075
            )
            / 0.035
        )
    )

    return _geometric_mean_probability(
        [
            elongation_probability,
            flatness_probability,
            extent_probability,
        ],
        1e-6,
    )


def child_shape_evidence(
    region_descriptions: dict[int, dict],
    merged_description: dict,
    epsilon: float,
) -> tuple[float, float]:
    child_probabilities = [
        single_cell_shape_probability(description)
        for description in region_descriptions.values()
    ]

    absolute_child_shape = _geometric_mean_probability(
        child_probabilities,
        epsilon,
    )

    mean_child_elongation = float(
        np.mean(
            [
                description["elongation"]
                for description in region_descriptions.values()
            ]
        )
    )
    mean_child_extent = float(
        np.mean(
            [
                description["extent"]
                for description in region_descriptions.values()
            ]
        )
    )

    elongation_improvement = float(
        expit(
            (
                float(merged_description["elongation"])
                - mean_child_elongation
            )
            / 0.40
        )
    )
    extent_improvement = float(
        expit(
            (
                mean_child_extent
                - float(merged_description["extent"])
            )
            / 0.080
        )
    )

    shape_improvement = _geometric_mean_probability(
        [
            elongation_improvement,
            extent_improvement,
        ],
        epsilon,
    )

    return absolute_child_shape, shape_improvement


def smooth_child_volume_probability(
    volume_fractions: np.ndarray,
) -> float:
    """Broad log-ratio distribution around equal-sized children."""

    count = len(volume_fractions)

    if count <= 1:
        return 1.0

    expected_fraction = 1.0 / count

    log_ratio = np.log(
        np.clip(
            volume_fractions / expected_fraction,
            1e-8,
            None,
        )
    )

    scores = np.exp(
        -0.5 * (log_ratio / 0.85) ** 2
    )

    return float(np.mean(scores))


def score_watershed_hypothesis(
    labels: np.ndarray,
    selected_peaks: list[dict],
    effective_peaks: list[dict],
    pair_table: pd.DataFrame,
    distance_um: np.ndarray,
    merged_description: dict,
    mask: np.ndarray,
    voxel_size_zyx: np.ndarray,
    config: dict,
) -> dict:
    epsilon = float(config["probability_epsilon"])

    hard_valid, hard_reasons = hard_gate_hypothesis(
        labels=labels,
        mask=mask,
        selected_peaks=selected_peaks,
        voxel_size_zyx=voxel_size_zyx,
        config=config,
    )

    if not hard_valid:
        return {
            "hard_valid": False,
            "hard_reasons": hard_reasons,
            "k": int(len(selected_peaks)),
            "labels": labels,
            "selected_peaks": selected_peaks,
            "log_posterior_unnormalized": -np.inf,
        }

    region_descriptions = describe_hypothesis_regions(
        labels,
        voxel_size_zyx,
    )

    k = len(region_descriptions)

    child_volumes = np.asarray(
        [
            region_descriptions[label]["volume_um3"]
            for label in sorted(region_descriptions)
        ],
        dtype=float,
    )

    total_volume = max(float(child_volumes.sum()), 1e-12)
    volume_fractions = child_volumes / total_volume
    minimum_child_fraction = float(volume_fractions.min())

    soft_fraction_center = {
        1: 0.50,
        2: 0.14,
        3: 0.075,
    }.get(k, 0.05)

    fragment_safety = (
        1.0
        if k == 1
        else float(
            expit(
                (
                    minimum_child_fraction
                    - soft_fraction_center
                )
                / 0.045
            )
        )
    )

    child_volume = smooth_child_volume_probability(
        volume_fractions
    )

    absolute_child_shape, shape_improvement = (
        child_shape_evidence(
            region_descriptions,
            merged_description,
            epsilon,
        )
    )

    pair_lookup = _pair_probability_lookup(pair_table)
    selected_lobe_probabilities = (
        selected_pair_probabilities(
            selected_peaks,
            pair_lookup,
        )
    )

    selected_peak_ids = {
        int(record["peak_id"])
        for record in selected_peaks
    }

    unexplained_lobe_probabilities: list[float] = []

    for candidate in effective_peaks:
        candidate_id = int(candidate["peak_id"])

        if candidate_id in selected_peak_ids:
            continue

        support_against_selected = [
            float(
                pair_lookup.get(
                    frozenset(
                        (
                            candidate_id,
                            int(selected["peak_id"]),
                        )
                    ),
                    0.0,
                )
            )
            for selected in selected_peaks
        ]

        unexplained_lobe_probabilities.append(
            _geometric_mean_probability(
                support_against_selected,
                epsilon,
            )
        )

    maximum_unexplained_lobe_probability = (
        max(unexplained_lobe_probabilities)
        if unexplained_lobe_probabilities
        else 0.0
    )

    coverage_support = float(
        np.clip(
            1.0 - maximum_unexplained_lobe_probability,
            epsilon,
            1.0,
        )
    )

    if k == 1:
        effective_pair_probabilities = (
            selected_pair_probabilities(
                effective_peaks,
                pair_lookup,
            )
        )

        maximum_pair_probability = (
            max(effective_pair_probabilities)
            if effective_pair_probabilities
            else 0.0
        )

        lobe_support = float(
            np.clip(
                1.0 - maximum_pair_probability,
                epsilon,
                1.0,
            )
        )
        neck_support = 0.75
        shape_improvement = 0.55
    else:
        lobe_support = _geometric_mean_probability(
            selected_lobe_probabilities,
            epsilon,
        )
        neck_support = 0.0

    marker_quality = _geometric_mean_probability(
        [
            float(record["persistence_score"])
            for record in selected_peaks
        ],
        epsilon,
    )

    interface_records: list[dict] = []
    all_interface_points: list[np.ndarray] = []
    pair_neck_probabilities: list[float] = []

    if k > 1:
        surfaces = {
            label: float(
                region_descriptions[label]["surface_area_um2"]
            )
            for label in region_descriptions
        }

        marker_depth_by_label = {
            marker_label: float(
                distance_um[
                    _peak_position_from_record(peak_record)
                ]
            )
            for marker_label, peak_record in enumerate(
                selected_peaks,
                start=1,
            )
        }

        for label_a, label_b in combinations(
            sorted(region_descriptions),
            2,
        ):
            interface = pair_interface_statistics(
                labels=labels,
                label_a=label_a,
                label_b=label_b,
                distance_um=distance_um,
                voxel_size_zyx=voxel_size_zyx,
            )

            if not interface["contact"]:
                continue

            smaller_peak_depth = max(
                min(
                    marker_depth_by_label[label_a],
                    marker_depth_by_label[label_b],
                ),
                1e-6,
            )

            saddle_ratio = float(
                interface["distance_median_um"]
                / smaller_peak_depth
            )

            neck_depth_probability = float(
                expit(
                    (
                        (1.0 - saddle_ratio) - 0.22
                    )
                    / 0.10
                )
            )

            normalized_interface_area = float(
                interface["interface_area_um2"]
                / max(
                    min(
                        surfaces[label_a],
                        surfaces[label_b],
                    ),
                    1e-6,
                )
            )

            interface_probability = float(
                expit(
                    (
                        0.16 - normalized_interface_area
                    )
                    / 0.045
                )
            )

            pair_neck_probability = (
                _geometric_mean_probability(
                    [
                        neck_depth_probability,
                        interface_probability,
                    ],
                    epsilon,
                )
            )

            pair_neck_probabilities.append(
                pair_neck_probability
            )
            all_interface_points.append(
                interface["interface_points_zyx"]
            )

            interface_records.append(
                {
                    "label_a": int(label_a),
                    "label_b": int(label_b),
                    "interface_area_um2": float(
                        interface["interface_area_um2"]
                    ),
                    "normalized_interface_area": (
                        normalized_interface_area
                    ),
                    "distance_median_um": float(
                        interface["distance_median_um"]
                    ),
                    "saddle_ratio": saddle_ratio,
                    "neck_depth_probability": (
                        neck_depth_probability
                    ),
                    "interface_probability": (
                        interface_probability
                    ),
                    "pair_neck_probability": (
                        pair_neck_probability
                    ),
                }
            )

        neck_support = _geometric_mean_probability(
            pair_neck_probabilities,
            epsilon,
        )

    component_probabilities = {
        "lobe_support": float(lobe_support),
        "coverage_support": float(coverage_support),
        "marker_quality": float(marker_quality),
        "neck_support": float(neck_support),
        "child_shape": float(absolute_child_shape),
        "shape_improvement": float(shape_improvement),
        "child_volume": float(child_volume),
        "fragment_safety": float(fragment_safety),
    }

    prior = float(
        config["hypothesis_priors"].get(k, epsilon)
    )

    log_likelihood = 0.0

    for name, weight in config[
        "hypothesis_evidence_weights"
    ].items():
        probability = float(
            np.clip(
                component_probabilities[name],
                epsilon,
                1.0,
            )
        )

        log_likelihood += float(weight) * np.log(
            probability
        )

    log_posterior_unnormalized = float(
        np.log(max(prior, epsilon))
        + log_likelihood
    )

    interface_points = (
        np.concatenate(all_interface_points, axis=0)
        if all_interface_points
        else np.empty((0, 3), dtype=float)
    )

    return {
        "hard_valid": True,
        "hard_reasons": [],
        "k": int(k),
        "labels": labels,
        "selected_peaks": selected_peaks,
        "selected_pair_lobe_probabilities": (
            selected_lobe_probabilities
        ),
        "region_descriptions": region_descriptions,
        "interface_records": interface_records,
        "interface_points_zyx": interface_points,
        "minimum_child_fraction": minimum_child_fraction,
        "maximum_unexplained_lobe_probability": float(
            maximum_unexplained_lobe_probability
        ),
        "prior_probability": prior,
        "log_likelihood": float(log_likelihood),
        "log_posterior_unnormalized": (
            log_posterior_unnormalized
        ),
        **component_probabilities,
    }

In [ ]:
# ============================================================
# Hierarchical H1 -> H2 -> H3 hypothesis generation
# ============================================================

def minimum_marker_separation_um(
    selected_peaks: list[dict],
    voxel_size_zyx: np.ndarray,
) -> float:
    if len(selected_peaks) < 2:
        return np.inf

    return float(
        min(
            physical_distance_between_points(
                _peak_position_from_record(first),
                _peak_position_from_record(second),
                voxel_size_zyx,
            )
            for first, second in combinations(
                selected_peaks,
                2,
            )
        )
    )


def combination_prescore(
    selected_peaks: list[dict],
    pair_lookup: dict[frozenset[int], float],
    epsilon: float,
) -> float:
    marker_quality = _geometric_mean_probability(
        [
            float(record["persistence_score"])
            for record in selected_peaks
        ],
        epsilon,
    )

    pair_probabilities = selected_pair_probabilities(
        selected_peaks,
        pair_lookup,
    )

    lobe_support = _geometric_mean_probability(
        pair_probabilities,
        epsilon,
    )

    return float(
        0.70 * lobe_support
        + 0.30 * marker_quality
    )


def evaluate_spatial_split_hypotheses(
    mask: np.ndarray,
    distance_um: np.ndarray,
    watershed_distance: np.ndarray,
    effective_peak_table: pd.DataFrame,
    pair_table: pd.DataFrame,
    merged_description: dict,
    voxel_size_zyx: np.ndarray,
    config: dict,
) -> tuple[list[dict], list[dict]]:
    candidate_pool = (
        effective_peak_table
        .head(int(config["max_candidate_peaks"]))
        .to_dict("records")
    )

    if not candidate_pool:
        raise ValueError(
            "No effective peak candidate remains after lobe collapsing."
        )

    pair_lookup = _pair_probability_lookup(pair_table)
    epsilon = float(config["probability_epsilon"])
    all_hypotheses: list[dict] = []

    # H1: retain the original connected component.
    unsplit_labels = mask.astype(np.int32)

    all_hypotheses.append(
        score_watershed_hypothesis(
            labels=unsplit_labels,
            selected_peaks=[candidate_pool[0]],
            effective_peaks=candidate_pool,
            pair_table=pair_table,
            distance_um=distance_um,
            merged_description=merged_description,
            mask=mask,
            voxel_size_zyx=voxel_size_zyx,
            config=config,
        )
    )

    # H2 is the primary split model.
    if len(candidate_pool) >= 2:
        ranked_pairs: list[
            tuple[float, list[dict]]
        ] = []

        for pair in combinations(candidate_pool, 2):
            selected = list(pair)

            if minimum_marker_separation_um(
                selected,
                voxel_size_zyx,
            ) < float(
                config["hard_min_marker_separation_um"]
            ):
                continue

            ranked_pairs.append(
                (
                    combination_prescore(
                        selected,
                        pair_lookup,
                        epsilon,
                    ),
                    selected,
                )
            )

        ranked_pairs.sort(
            key=lambda item: item[0],
            reverse=True,
        )

        for prescore, selected in ranked_pairs[
            : int(config["max_combinations_per_k"])
        ]:
            labels = build_watershed_hypothesis(
                mask,
                watershed_distance,
                selected,
            )

            result = score_watershed_hypothesis(
                labels=labels,
                selected_peaks=selected,
                effective_peaks=candidate_pool,
                pair_table=pair_table,
                distance_um=distance_um,
                merged_description=merged_description,
                mask=mask,
                voxel_size_zyx=voxel_size_zyx,
                config=config,
            )
            result["combination_prescore"] = float(
                prescore
            )
            all_hypotheses.append(result)

    # H3 is generated only after three independent effective lobes survive.
    if (
        int(config["max_cells"]) >= 3
        and len(candidate_pool) >= 3
    ):
        ranked_triples: list[
            tuple[float, list[dict]]
        ] = []

        for triple in combinations(candidate_pool, 3):
            selected = list(triple)

            if minimum_marker_separation_um(
                selected,
                voxel_size_zyx,
            ) < float(
                config["hard_min_marker_separation_um"]
            ):
                continue

            pair_probabilities = selected_pair_probabilities(
                selected,
                pair_lookup,
            )

            if len(pair_probabilities) != 3:
                continue

            if min(pair_probabilities) < float(
                config[
                    "k3_generation_min_pair_probability"
                ]
            ):
                continue

            geometric_probability = (
                _geometric_mean_probability(
                    pair_probabilities,
                    epsilon,
                )
            )

            if geometric_probability < float(
                config[
                    "k3_generation_min_geometric_probability"
                ]
            ):
                continue

            ranked_triples.append(
                (
                    combination_prescore(
                        selected,
                        pair_lookup,
                        epsilon,
                    ),
                    selected,
                )
            )

        ranked_triples.sort(
            key=lambda item: item[0],
            reverse=True,
        )

        for prescore, selected in ranked_triples[
            : int(config["max_combinations_per_k"])
        ]:
            labels = build_watershed_hypothesis(
                mask,
                watershed_distance,
                selected,
            )

            result = score_watershed_hypothesis(
                labels=labels,
                selected_peaks=selected,
                effective_peaks=candidate_pool,
                pair_table=pair_table,
                distance_um=distance_um,
                merged_description=merged_description,
                mask=mask,
                voxel_size_zyx=voxel_size_zyx,
                config=config,
            )
            result["combination_prescore"] = float(
                prescore
            )
            all_hypotheses.append(result)

    valid_hypotheses = [
        result
        for result in all_hypotheses
        if bool(result.get("hard_valid", False))
        and np.isfinite(
            result["log_posterior_unnormalized"]
        )
    ]

    if not valid_hypotheses:
        raise RuntimeError(
            "Every spatial hypothesis failed the broad safety gates."
        )

    best_by_k: list[dict] = []

    for k in sorted(
        {
            int(result["k"])
            for result in valid_hypotheses
        }
    ):
        best_by_k.append(
            max(
                (
                    result
                    for result in valid_hypotheses
                    if int(result["k"]) == k
                ),
                key=lambda result: (
                    result["log_posterior_unnormalized"]
                ),
            )
        )

    log_values = np.asarray(
        [
            result["log_posterior_unnormalized"]
            for result in best_by_k
        ],
        dtype=float,
    )

    normalized_logs = log_values - logsumexp(log_values)

    for result, normalized_log in zip(
        best_by_k,
        normalized_logs,
    ):
        result["posterior_probability"] = float(
            np.exp(normalized_log)
        )

    return best_by_k, all_hypotheses

In [ ]:
# ============================================================
# Posterior decision and whole-volume component integration
# ============================================================

def choose_hierarchical_hypothesis(
    best_hypotheses: list[dict],
    config: dict,
) -> dict:
    hypothesis_by_k = {
        int(result["k"]): result
        for result in best_hypotheses
    }

    h1 = hypothesis_by_k[1]
    h2 = hypothesis_by_k.get(2)
    h3 = hypothesis_by_k.get(3)

    epsilon = float(config["probability_epsilon"])
    p1 = float(h1["posterior_probability"])
    p2 = (
        float(h2["posterior_probability"])
        if h2 is not None
        else 0.0
    )
    p3 = (
        float(h3["posterior_probability"])
        if h3 is not None
        else 0.0
    )

    h2_conditional_probability = (
        p2 / max(p1 + p2, epsilon)
        if h2 is not None
        else 0.0
    )
    h2_odds_vs_h1 = (
        p2 / max(p1, epsilon)
        if h2 is not None
        else 0.0
    )

    h2_accepted = bool(
        h2 is not None
        and h2_conditional_probability
        >= float(config["h2_min_conditional_probability"])
        and h2_odds_vs_h1
        >= float(config["h2_min_odds_vs_h1"])
    )

    if h2_accepted:
        chosen_hypothesis = h2
        decision_status = "accepted_two_cell_split"
    else:
        chosen_hypothesis = h1

        if (
            h2 is not None
            and h2_conditional_probability
            > float(
                config["h1_confident_conditional_probability"]
            )
        ):
            decision_status = "uncertain_no_split"
        else:
            decision_status = "accepted_single"

    h3_conditional_probability = (
        p3 / max(p2 + p3, epsilon)
        if h3 is not None and h2 is not None
        else 0.0
    )
    h3_odds_vs_h2 = (
        p3 / max(p2, epsilon)
        if h3 is not None and h2 is not None
        else 0.0
    )

    h3_accepted = bool(
        h2_accepted
        and h3 is not None
        and h3_conditional_probability
        >= float(config["h3_min_conditional_probability"])
        and h3_odds_vs_h2
        >= float(config["h3_min_odds_vs_h2"])
    )

    if h3_accepted:
        chosen_hypothesis = h3
        decision_status = "accepted_three_cell_split"

    return {
        "chosen_hypothesis": chosen_hypothesis,
        "decision_status": decision_status,
        "split_accepted": int(chosen_hypothesis["k"]) > 1,
        "h2_conditional_probability": h2_conditional_probability,
        "h2_odds_vs_h1": h2_odds_vs_h1,
        "h3_conditional_probability": h3_conditional_probability,
        "h3_odds_vs_h2": h3_odds_vs_h2,
    }


def analyze_component_crop(
    component_mask: np.ndarray,
    voxel_size_zyx: np.ndarray,
    config: dict,
) -> dict:
    """Run spatial inference on one tight component crop."""

    component_mask = np.asarray(component_mask, dtype=bool)

    if component_mask.ndim != 3 or not component_mask.any():
        raise ValueError("component_mask must be a non-empty 3-D mask.")

    padding = int(config["component_padding_voxels"])
    padded_mask = np.pad(
        component_mask,
        padding,
        mode="constant",
        constant_values=False,
    )

    (
        raw_distance,
        merge_tree_distance,
        watershed_distance,
        peak_table,
    ) = detect_persistent_distance_peaks(
        mask=padded_mask,
        voxel_size_zyx=voxel_size_zyx,
        config=config,
    )

    pair_table = build_peak_pair_table(
        peak_table=peak_table,
        mask=padded_mask,
        merge_tree_distance=merge_tree_distance,
        voxel_size_zyx=voxel_size_zyx,
        config=config,
    )

    annotated_peak_table, effective_peak_table = (
        collapse_same_lobe_peaks(
            peak_table=peak_table,
            pair_table=pair_table,
            config=config,
        )
    )

    merged_description = describe_cell_mask(
        padded_mask,
        voxel_size_zyx,
    )

    best_hypotheses, all_hypotheses = (
        evaluate_spatial_split_hypotheses(
            mask=padded_mask,
            distance_um=raw_distance,
            watershed_distance=watershed_distance,
            effective_peak_table=effective_peak_table,
            pair_table=pair_table,
            merged_description=merged_description,
            voxel_size_zyx=voxel_size_zyx,
            config=config,
        )
    )

    decision = choose_hierarchical_hypothesis(
        best_hypotheses,
        config,
    )
    chosen = decision["chosen_hypothesis"]

    inner = tuple(
        slice(padding, -padding)
        if padding > 0
        else slice(None)
        for _ in range(3)
    )
    chosen_labels = np.asarray(
        chosen["labels"][inner],
        dtype=np.int32,
    )

    selected_positions = [
        tuple(
            int(coordinate) - padding
            for coordinate in _peak_position_from_record(record)
        )
        for record in chosen["selected_peaks"]
    ]

    if np.any(chosen_labels[component_mask] <= 0):
        raise RuntimeError(
            "Chosen local hypothesis does not cover the source component."
        )
    if np.any(chosen_labels[~component_mask] != 0):
        raise RuntimeError(
            "Chosen local hypothesis extends outside the source component."
        )

    return {
        "labels": chosen_labels,
        "selected_positions_zyx": selected_positions,
        "peak_table": peak_table,
        "annotated_peak_table": annotated_peak_table,
        "effective_peak_table": effective_peak_table,
        "pair_table": pair_table,
        "best_hypotheses": best_hypotheses,
        "all_hypotheses": all_hypotheses,
        **decision,
    }


def _posterior_by_k(
    best_hypotheses: list[dict],
    k: int,
) -> float:
    for result in best_hypotheses:
        if int(result["k"]) == int(k):
            return float(result["posterior_probability"])
    return 0.0


instance_labels = np.zeros(
    binary_mask.shape,
    dtype=np.int32,
)
markers = np.zeros(
    binary_mask.shape,
    dtype=np.int32,
)

component_diagnostic_records: list[dict] = []
hypothesis_diagnostic_records: list[dict] = []
instance_mapping_records: list[dict] = []

next_instance_label = 1
pipeline_start = time.perf_counter()

for component_id, component_slice in enumerate(
    component_slices,
    start=1,
):
    if component_slice is None:
        continue

    local_component = (
        component_labels[component_slice] == component_id
    )
    source_voxels = int(np.count_nonzero(local_component))

    result = None

    try:
        result = analyze_component_crop(
            component_mask=local_component,
            voxel_size_zyx=VOXEL_SIZE,
            config=SPLIT_CONFIG,
        )

        local_labels = result["labels"]
        selected_positions = result[
            "selected_positions_zyx"
        ]
        best_hypotheses = result["best_hypotheses"]
        chosen = result["chosen_hypothesis"]
        decision_status = result["decision_status"]
        error_message = ""

        raw_peak_count = len(result["annotated_peak_table"])
        effective_lobe_count = len(
            result["effective_peak_table"]
        )

    except Exception as error:
        # Safe production fallback: preserve the Stage 3 component as one
        # instance and choose its deepest point in the retained global EDT.
        local_labels = local_component.astype(np.int32)
        local_distance = distance_smooth[component_slice]
        masked_distance = np.where(
            local_component,
            local_distance,
            -np.inf,
        )
        fallback_position = tuple(
            int(value)
            for value in np.unravel_index(
                int(np.argmax(masked_distance)),
                masked_distance.shape,
            )
        )
        selected_positions = [fallback_position]
        best_hypotheses = []
        chosen = {"k": 1}
        decision_status = "fallback_single"
        error_message = (
            f"{type(error).__name__}: {error}"
        )
        raw_peak_count = 0
        effective_lobe_count = 1

    local_positive_labels = [
        int(value)
        for value in np.unique(local_labels)
        if int(value) > 0
    ]

    global_label_by_local: dict[int, int] = {}
    target_view = instance_labels[component_slice]

    for local_label in local_positive_labels:
        global_label = next_instance_label
        next_instance_label += 1
        global_label_by_local[local_label] = global_label

        region = local_labels == local_label
        target_view[region] = global_label

        instance_mapping_records.append(
            {
                "component_id": component_id,
                "local_child_label": local_label,
                "instance_id": global_label,
                "voxel_count": int(np.count_nonzero(region)),
                "decision_status": decision_status,
            }
        )

    if len(selected_positions) != len(local_positive_labels):
        raise RuntimeError(
            f"Component {component_id}: marker count "
            f"{len(selected_positions)} does not match child count "
            f"{len(local_positive_labels)}."
        )

    starts = np.asarray(
        [axis_slice.start for axis_slice in component_slice],
        dtype=int,
    )

    global_marker_positions: list[tuple[int, int, int]] = []

    for local_label, local_position in zip(
        local_positive_labels,
        selected_positions,
    ):
        local_position_array = np.asarray(
            local_position,
            dtype=int,
        )
        global_position = tuple(
            int(value)
            for value in starts + local_position_array
        )
        global_label = global_label_by_local[local_label]

        if not binary_mask[global_position]:
            raise RuntimeError(
                f"Component {component_id}: marker {global_position} "
                "lies outside the foreground mask."
            )

        markers[global_position] = global_label
        global_marker_positions.append(global_position)

    posterior_h1 = _posterior_by_k(best_hypotheses, 1)
    posterior_h2 = _posterior_by_k(best_hypotheses, 2)
    posterior_h3 = _posterior_by_k(best_hypotheses, 3)

    component_diagnostic_records.append(
        {
            "component_id": component_id,
            "source_voxels": source_voxels,
            "raw_peak_count": raw_peak_count,
            "effective_lobe_count": effective_lobe_count,
            "chosen_k": len(local_positive_labels),
            "decision_status": decision_status,
            "split_accepted": len(local_positive_labels) > 1,
            "posterior_h1": posterior_h1,
            "posterior_h2": posterior_h2,
            "posterior_h3": posterior_h3,
            "h2_conditional_probability": float(
                result["h2_conditional_probability"]
            ) if result is not None and not error_message else 0.0,
            "h2_odds_vs_h1": float(
                result["h2_odds_vs_h1"]
            ) if result is not None and not error_message else 0.0,
            "h3_conditional_probability": float(
                result["h3_conditional_probability"]
            ) if result is not None and not error_message else 0.0,
            "h3_odds_vs_h2": float(
                result["h3_odds_vs_h2"]
            ) if result is not None and not error_message else 0.0,
            "marker_positions_zyx": json.dumps(
                global_marker_positions
            ),
            "bbox_z0": int(component_slice[0].start),
            "bbox_z1": int(component_slice[0].stop),
            "bbox_y0": int(component_slice[1].start),
            "bbox_y1": int(component_slice[1].stop),
            "bbox_x0": int(component_slice[2].start),
            "bbox_x1": int(component_slice[2].stop),
            "error": error_message,
        }
    )

    for hypothesis in best_hypotheses:
        hypothesis_diagnostic_records.append(
            {
                "component_id": component_id,
                "k": int(hypothesis["k"]),
                "posterior_probability": float(
                    hypothesis["posterior_probability"]
                ),
                "prior_probability": float(
                    hypothesis["prior_probability"]
                ),
                "log_likelihood": float(
                    hypothesis["log_likelihood"]
                ),
                "lobe_support": float(
                    hypothesis["lobe_support"]
                ),
                "coverage_support": float(
                    hypothesis["coverage_support"]
                ),
                "marker_quality": float(
                    hypothesis["marker_quality"]
                ),
                "neck_support": float(
                    hypothesis["neck_support"]
                ),
                "child_shape": float(
                    hypothesis["child_shape"]
                ),
                "shape_improvement": float(
                    hypothesis["shape_improvement"]
                ),
                "child_volume": float(
                    hypothesis["child_volume"]
                ),
                "fragment_safety": float(
                    hypothesis["fragment_safety"]
                ),
                "minimum_child_fraction": float(
                    hypothesis["minimum_child_fraction"]
                ),
                "selected_peak_ids": json.dumps(
                    [
                        int(record["peak_id"])
                        for record in hypothesis["selected_peaks"]
                    ]
                ),
            }
        )

    progress_every = int(
        SPLIT_CONFIG["progress_every_components"]
    )
    if (
        component_id == 1
        or component_id % progress_every == 0
        or component_id == num_components
    ):
        elapsed = time.perf_counter() - pipeline_start
        print(
            f"Processed {component_id}/{num_components} components | "
            f"instances={next_instance_label - 1} | "
            f"elapsed={elapsed:.1f}s"
        )

component_split_diagnostics = pd.DataFrame(
    component_diagnostic_records
)
hypothesis_diagnostics = pd.DataFrame(
    hypothesis_diagnostic_records
)
instance_component_map = pd.DataFrame(
    instance_mapping_records
)

if np.any(instance_labels[binary_mask] <= 0):
    raise RuntimeError(
        "Final instance labels do not cover the complete binary mask."
    )
if np.any(instance_labels[~binary_mask] != 0):
    raise RuntimeError(
        "Final instance labels extend outside the binary mask."
    )

print("Final marker count:", int(np.count_nonzero(markers)))
print("Number of instances:", int(instance_labels.max()))


## Inspect probabilistic split decisions

In [ ]:
# ============================================================
# Pipeline diagnostics
# ============================================================

decision_counts = (
    component_split_diagnostics["decision_status"]
    .value_counts()
    .rename_axis("decision_status")
    .to_frame("component_count")
)

display(decision_counts)

accepted_splits = (
    component_split_diagnostics[
        component_split_diagnostics["split_accepted"]
    ]
    .sort_values(
        ["chosen_k", "posterior_h2", "posterior_h3"],
        ascending=[False, False, False],
    )
    .reset_index(drop=True)
)

print("Input connected components:", num_components)
print("Output instances:", int(instance_labels.max()))
print("Accepted split components:", len(accepted_splits))
print(
    "Two-cell splits:",
    int(
        (
            component_split_diagnostics["chosen_k"] == 2
        ).sum()
    ),
)
print(
    "Three-cell splits:",
    int(
        (
            component_split_diagnostics["chosen_k"] == 3
        ).sum()
    ),
)
print(
    "Uncertain components retained unsplit:",
    int(
        (
            component_split_diagnostics["decision_status"]
            == "uncertain_no_split"
        ).sum()
    ),
)
print(
    "Fallback components:",
    int(
        (
            component_split_diagnostics["decision_status"]
            == "fallback_single"
        ).sum()
    ),
)

if not accepted_splits.empty:
    display(
        accepted_splits[
            [
                "component_id",
                "source_voxels",
                "raw_peak_count",
                "effective_lobe_count",
                "chosen_k",
                "posterior_h1",
                "posterior_h2",
                "posterior_h3",
                "h2_conditional_probability",
                "h3_conditional_probability",
                "marker_positions_zyx",
            ]
        ].head(30).round(4)
    )

fallback_rows = component_split_diagnostics[
    component_split_diagnostics["decision_status"]
    == "fallback_single"
]

if not fallback_rows.empty:
    print("Components preserved by the safety fallback:")
    display(
        fallback_rows[
            ["component_id", "source_voxels", "error"]
        ]
    )


## Visualize final instances

In [ ]:
z = instance_labels.shape[0] // 2

fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(binary_mask[z], cmap="gray")
ax[0].set_title("Binary Mask")

ax[1].imshow(instance_labels[z], cmap="nipy_spectral")
marker_plane = markers[z] > 0
if np.any(marker_plane):
    marker_y, marker_x = np.nonzero(marker_plane)
    ax[1].scatter(
        marker_x,
        marker_y,
        s=12,
        facecolors="none",
        edgecolors="white",
        linewidths=0.8,
    )
ax[1].set_title("Probabilistic Watershed Instances")

for axis in ax:
    axis.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
z = instance_labels.shape[0] // 2

boundaries = find_boundaries(
    instance_labels,
    mode="outer",
)

plt.figure(figsize=(8, 8))
plt.imshow(binary_mask[z], cmap="gray")
plt.contour(boundaries[z], colors="red", linewidths=1)
plt.title("Probabilistic Watershed Boundaries")
plt.axis("off")
plt.show()


## Instance-size statistics

In [ ]:
labels = np.unique(instance_labels)
labels = labels[labels != 0]

sizes = ndimage.sum(
    np.ones_like(instance_labels, dtype=np.uint8),
    labels=instance_labels,
    index=labels,
)

print("Instances :", len(labels))
print("Smallest  :", float(sizes.min()))
print("Largest   :", float(sizes.max()))
print("Median    :", float(np.median(sizes)))
print("Mean      :", float(np.mean(sizes)))


## Save results

In [ ]:
OUTPUT_DIR = Path(
    "../data/sample/processed/stage_4_instance_segmentation"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.save(
    OUTPUT_DIR / "instance_labels.npy",
    instance_labels,
)
np.save(
    OUTPUT_DIR / "markers.npy",
    markers,
)

component_split_diagnostics.to_csv(
    OUTPUT_DIR / "component_split_diagnostics.csv",
    index=False,
)
hypothesis_diagnostics.to_csv(
    OUTPUT_DIR / "split_hypothesis_diagnostics.csv",
    index=False,
)
instance_component_map.to_csv(
    OUTPUT_DIR / "instance_component_map.csv",
    index=False,
)

run_summary = {
    "sample_name": SAMPLE_NAME,
    "input_components": int(num_components),
    "output_instances": int(instance_labels.max()),
    "accepted_two_cell_splits": int(
        (component_split_diagnostics["chosen_k"] == 2).sum()
    ),
    "accepted_three_cell_splits": int(
        (component_split_diagnostics["chosen_k"] == 3).sum()
    ),
    "uncertain_no_split": int(
        (
            component_split_diagnostics["decision_status"]
            == "uncertain_no_split"
        ).sum()
    ),
    "fallback_single": int(
        (
            component_split_diagnostics["decision_status"]
            == "fallback_single"
        ).sum()
    ),
    "voxel_size_zyx_um": VOXEL_SIZE.tolist(),
    "split_config": SPLIT_CONFIG,
}

with (
    OUTPUT_DIR / "instance_segmentation_summary.json"
).open("w", encoding="utf-8") as file:
    json.dump(
        run_summary,
        file,
        indent=2,
    )

print(f"Saved to {OUTPUT_DIR}")
